In [1]:
!pip install -q transformers datasets trl accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/751.0 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━ 317.4/751.0 kB 9.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 11.6 MB/s eta 0:00:00


In [2]:
# import torch
# from datasets import load_dataset
# from transformers import GPT2Tokenizer, GPT2LMHeadModel
# from trl import SFTTrainer, SFTConfig

# # 1. KHỞI TẠO TOKENIZER VÀ THÊM TỪ VỰNG MỚI
# model_name = "gpt2-medium"
# tokenizer = GPT2Tokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<tools>', '</tools>', '<tool_call>', '</tool_call>']}
# num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
# print(f"Đã thêm {num_added_toks} tokens đặc biệt vào Tokenizer.")

# # 2. KHỞI TẠO MÔ HÌNH VÀ RESIZE EMBEDDINGS
# model = GPT2LMHeadModel.from_pretrained(model_name)
# model.resize_token_embeddings(len(tokenizer))

# data_files = {
#     "train": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/train_tool_data_copyfix.jsonl",
#     "validation": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/valid_tool_data_copyfix.jsonl"
# }

# dataset = load_dataset("json", data_files=data_files)

# train_dataset = dataset["train"]
# val_dataset = dataset["validation"]
# print(f"Số dòng Train: {len(train_dataset)} | Số dòng Val: {len(val_dataset)}")

# # 4. CẤU HÌNH HUẤN LUYỆN 
# training_args = SFTConfig(
#     output_dir="./gpt2-medium-agent",
#     per_device_train_batch_size=2,   
#     gradient_accumulation_steps=4,   
#     learning_rate=3e-5,              
#     num_train_epochs=5,                # Tăng lên 5 vòng để học kỹ tham số
#     logging_steps=50,
    
#     # --- CẤU HÌNH ĐÁNH GIÁ (VALIDATION) ---
#     eval_strategy="steps",             
#     eval_steps=500,                    
#     save_steps=500,                    
#     load_best_model_at_end=True,       
#     metric_for_best_model="eval_loss", 
#     save_total_limit=2,                
    
#     fp16=True,                       
#     optim="adamw_torch",
#     report_to="none",
    
#     dataset_text_field="text",       
#     max_length=512,                  
#     packing=False                    
# )

# # 5. CHẠY SFT TRAINER
# trainer = SFTTrainer(
#     model=model,
#     processing_class=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,          # Nạp tập Val vào "trường thi"
#     args=training_args,
# )

# print("Bắt đầu quá trình huấn luyện có kiểm duyệt (Validation)...")
# trainer.train()

# # 6. LƯU MÔ HÌNH XUẤT SẮC NHẤT
# model.save_pretrained("./final_gpt2_agent_v2")
# tokenizer.save_pretrained("./final_gpt2_agent_v2")
# print("Đã lưu thành công phiên bản GPT-2 xịn nhất!")

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from trl import SFTTrainer, SFTConfig

# Đường dẫn checkpoint cũ trong mục input của bạn
RESUME_CHECKPOINT_PATH = "/kaggle/input/models/hykhangg/newdata-gpt2-medium-finetune/transformers/default/1/gpt2-medium-agent/checkpoint-11500"

# 1. KHỞI TẠO TOKENIZER TỪ CHECKPOINT CŨ
if os.path.exists(RESUME_CHECKPOINT_PATH):
    print(f"[INFO] Load Tokenizer trực tiếp từ Checkpoint...")
    tokenizer = GPT2Tokenizer.from_pretrained(RESUME_CHECKPOINT_PATH)
else:
    print(f"[WARNING] Không thấy checkpoint, dùng cấu hình mặc định...")
    model_name = "gpt2-medium"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<tools>', '</tools>', '<tool_call>', '</tool_call>']}
    tokenizer.add_special_tokens(special_tokens_dict)

# 2. KHỞI TẠO MÔ HÌNH THẲNG TỪ CHECKPOINT TRONG INPUT
if os.path.exists(RESUME_CHECKPOINT_PATH):
    print(f"[INFO] Load Mô hình trực tiếp từ Checkpoint (Đảm bảo đủ lm_head)...")
    model = GPT2LMHeadModel.from_pretrained(RESUME_CHECKPOINT_PATH)
else:
    print(f"[INFO] Khởi tạo mô hình mới từ gpt2-medium...")
    model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
    model.resize_token_embeddings(len(tokenizer))

# XEM THỬ ĐẦU RA: Kiểm tra chắc chắn lớp lm_head đã được load với kích thước mở rộng
print(f"[CHECK] Kích thước lm_head hiện tại: {model.lm_head.weight.shape}")

# 3. TẢI DATASET
data_files = {
    "train": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/train_tool_data_copyfix.jsonl",
    "validation": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/valid_tool_data_copyfix.jsonl"
}
dataset = load_dataset("json", data_files=data_files)
train_dataset = dataset["train"]
val_dataset = dataset["validation"]
print(f"Số dòng Train: {len(train_dataset)} | Số dòng Val: {len(val_dataset)}")

# 4. CẤU HÌNH HUẤN LUYỆN 
training_args = SFTConfig(
    output_dir="./gpt2-medium-agent",       # Nơi lưu các checkpoint mới phát sinh (working)
    per_device_train_batch_size=2,   
    gradient_accumulation_steps=4,   
    learning_rate=3e-5,              
    num_train_epochs=5,                
    logging_steps=50,
    
    # --- CẤU HÌNH ĐÁNH GIÁ (VALIDATION) ---
    eval_strategy="steps",             
    eval_steps=500,                    
    save_steps=500,                    
    load_best_model_at_end=True,       
    metric_for_best_model="eval_loss", 
    save_total_limit=2,                
    
    fp16=True,                       
    optim="adamw_torch",
    report_to="none",
    
    dataset_text_field="text",       
    max_length=512,                  
    packing=False                    
)

# 5. KHỞI TẠO SFT TRAINER
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("\nBắt đầu quá trình huấn luyện tiếp tục (đã bỏ qua bước nạp đè của Trainer)...")
# Bằng cách không truyền resume_from_checkpoint, Trainer sẽ coi đây là chu kỳ train mới 
# chạy trên nền trọng số chuẩn chỉnh đã nạp sẵn từ bước 2, loại bỏ hoàn toàn lỗi lạc mất key!
trainer.train()

# 6. LƯU MÔ HÌNH XUẤT SẮC NHẤT
trainer.save_model("./final_gpt2_agent_v2")
tokenizer.save_pretrained("./final_gpt2_agent_v2")
print("\n[SUCCESS] Đã huấn luyện xong hoàn toàn và lưu phiên bản tối ưu nhất!")

[INFO] Load Tokenizer trực tiếp từ Checkpoint...


[INFO] Load Mô hình trực tiếp từ Checkpoint (Đảm bảo đủ lm_head)...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[CHECK] Kích thước lm_head hiện tại: torch.Size([50263, 1024])


Generating train split: 0 examples [00:00, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]